In [1]:
import cell2mol
import numpy as np
import os

In [2]:
infopath = "INOVAL/INOVAL.info"

In [3]:
from cell2mol.read_write import readinfo
from cell2mol.classes import cell
from cell2mol.cell_reconstruction import classify_fragments, fragments_reconstruct
from cell2mol.charge_assignment import *

In [4]:
name = "INOVAL"

In [6]:
debug=0

In [7]:
print(f"INITIATING cell object from input") 

# Reads reference molecules from info file, as well as labels and coordinates
labels, pos, ref_labels, ref_fracs, cellvec, cellparam = readinfo(infopath)

# Initiates cell
newcell = cell(name, labels, pos, cellvec, cellparam)
# # Loads the reference molecules and checks_missing_H
newcell.get_reference_molecules(ref_labels, ref_fracs, debug=debug) 
newcell.assess_errors()

INITIATING cell object from input
MOLECULE.SPLIT COMPLEX: labels=['Fe', 'Cl', 'Cl', 'Cl', 'O', 'H', 'H']
MOLECULE.SPLIT COMPLEX: metal_idx=[0]
MOLECULE.SPLIT COMPLEX: rest_idx=[1, 2, 3, 4, 5, 6]
SPLIT COMPLEX: rest labels: ['Cl', 'Cl', 'Cl', 'O', 'H', 'H']
SPLIT COMPLEX: rest coord: [[4.8139984268383635, 1.1805652875553925, 3.333073124178108], [7.250636970581888, 1.62258466795451, 0.45999712960445105], [4.856882530390316, 4.427793920077216, 1.030069539186763], [3.7025349960199416, 1.3810959182656035, 0.13850742300244523], [3.879901531654197, 0.5498751049594834, -0.0380260058105337], [2.9430798919008274, 1.49644435785722, 0.4002737453740389]]
SPLIT COMPLEX: rest indices: [1, 2, 3, 4, 5, 6]
SPLIT COMPLEX: rest radii: [1.02, 1.02, 1.02, 0.66, 0.31, 0.31]
SPLIT COMPLEX: splitting species with 6 atoms in block
labels=['Cl', 'Cl', 'Cl', 'O', 'H', 'H'] 6
indices=None
SPLIT COMPLEX: received 4 blocks
PREPARING BLOCK: [3, 4, 5]
CREATING LIGAND: H2-O
PREPARING BLOCK: [2]
CREATING LIGAND: Cl
PREP

In [9]:
from cell2mol.missingH import get_missingH_from_adjacency
Missing_H_in_C = False
Missing_H_in_CoordWater = False
ismissingH = False
Warning = False

# List of Metal Atoms for which O atoms might appear connected directly.
Exceptions_for_CoordWater = ["Re", "V", "Mo", "W"]

if debug >= 2: print("")
if debug >= 2: print("##################")
if debug >= 2: print("Checking Missing H")
if debug >= 2: print("##################")
for idx, ref in enumerate(newcell.refmoleclist):
    if not ref.iscomplex:
        if ref.natoms == 1 and "O" in ref.labels: 
            Missing_H_in_CoordWater = True
            if debug >= 2: print(f"WARNING found isolated O atom in the cell. This tends to be a water with missing H, so stopping")
        else:
            for kdx, a in enumerate(ref.atoms):
                if not hasattr(a,"adjacency"): continue 
                if a.label == "C":
                    bonded_atom_coord = []
                    for adj in a.adjacency:
                        bonded_atom_coord.append(ref.coord[adj])
                    ismissingH, report = get_missingH_from_adjacency(a.atnum, a.coord, bonded_atom_coord)
                    if ismissingH:
                        if debug >= 2: print("")
                        if debug >= 2: print(f"WARNING in Missing H function for: {ref.type}, {idx}, {ref.labels}")
                        if debug >= 2: print(f"C Atom {kdx} has missing H atoms")
                        if debug >= 2: print(report)
                        Missing_H_in_C = True
    else:
        for jdx, lig in enumerate(ref.ligands):
            if lig.natoms == 1 and "O" in lig.labels and lig.denticity <= 1:
                if any(m.label in Exceptions_for_CoordWater for m in lig.metalatoms): pass
                else:
                    Missing_H_in_CoordWater = True
                    if debug >= 2: print("")
                    if debug >= 2: print("WARNING in Missing H function for ligand",lig.natoms,lig.labels)
            else:
                for kdx, a in enumerate(lig.atoms):
                    if a.label == "C" and a.mconnec == 0:
                        bonded_atom_coord = []
                        print(a.label, a.adjacency, len(lig.coord))
                        for adj in a.adjacency:
                            bonded_atom_coord.append(lig.coord[adj])
                        ismissingH, report = get_missingH_from_adjacency(a.atnum, a.coord, bonded_atom_coord)
                        if ismissingH:
                            if debug >= 2: print("")
                            if debug >= 2: print(f"WARNING in Missing H function for: {ref.type}, {idx}, {jdx}, {lig.labels}")
                            if debug >= 2: print(f"Atom {kdx} has missing H atoms")
                            if debug >= 2: print(report)
                            Missing_H_in_C = True

In [7]:
newcell.check_missing_H(debug=debug)                                     

False

In [8]:
newcell.reconstruct(debug=debug) 


##############################################
FRAG_RECONSTRUCT. 22 molecules submitted to SEQUENTIAL with Heavy
##############################################
FRAG_RECONSTRUCT. 0 molecules and 8 fragments out of SEQUENTIAL with Heavy
FRAG_RECONSTRUCT. 24 molecules submitted to sequential with All
FINISHED succesfully
FRAG_RECONSTRUCT. No remaining Molecules after Hydrogen reconstruction
MOLECULE.SPLIT COMPLEX: labels=['Fe', 'Fe', 'Cl', 'Cl', 'Cl', 'Cl', 'Cl', 'Cl']
MOLECULE.SPLIT COMPLEX: metal_idx=[0, 1]
MOLECULE.SPLIT COMPLEX: rest_idx=[2, 3, 4, 5, 6, 7]
SPLIT COMPLEX: rest labels: ['Cl', 'Cl', 'Cl', 'Cl', 'Cl', 'Cl']
SPLIT COMPLEX: rest coord: [[8.504325, 4.9441406, 8.6069974], [12.3534909, 10.2389616, 7.2768814], [10.0814801, 8.2617062, 9.5185732], [10.7763357, 6.921396, 6.3653056], [7.4116689, 8.4886261, 6.9897009], [13.4461469, 6.6944761, 8.8941779]]
SPLIT COMPLEX: rest indices: [2, 3, 4, 5, 6, 7]
SPLIT COMPLEX: rest radii: [1.02, 1.02, 1.02, 1.02, 1.02, 1.02]
SPLIT COMPLEX: sp

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 8
  Formula                      = Cl6-Fe2
  Has Adjacency Matrix         = YES
  Origin                       = cell.reconstruct
  Number of Ligands            = 6
  Number of Metals             = 2
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 15
  Formula                      = H10-C4-O
  Has Adjacency Matrix         = YES
  Origin                       = cell.reconstruct
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type    

In [9]:
newcell.error_reconstruction

False

In [10]:
newcell.is_fragmented

False

In [11]:
newcell.get_unique_species(debug=debug)

[------------- Cell2mol LIGAND Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = ligand
  Number of Atoms              = 1
  Formula                      = Cl
  Has Adjacency Matrix         = YES
  Origin                       = split_complex
 ---------------------------------------------------,
 ------------- Cell2mol METAL Object --------------
  Version                      = 0.1
  Type                         = atom
  Sub-Type                     = metal
  Label                        = Fe
  Atomic Number                = 26
  Index in Molecule            = 0
  Metal Adjacency (mconnec)    = 5
  Regular Adjacencies (connec) = 5
  Coordination Sphere Formula  = Cl4-Fe
 ----------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number 

In [12]:
selected_cs = []
for idx, spec in enumerate(newcell.unique_species):
#     print("doing", spec)
    if spec.subtype != "metal":
        spec.get_protonation_states()
        spec.get_possible_cs()
        print(spec.formula)
        selected_cs.append(list([cs.corr_total_charge for cs in spec.possible_cs]))
        print(f"{spec.protonation_states=}")
#         for prot in spec.protonation_states :
#              for sub_spec, sub_prot in zip(spec.coord, prot.coords):
#                 print(sub_spec, sub_prot, (sub_spec == sub_prot)) 
#         print("==========")     
#         for cs in spec.possible_cs :
#              for sub_spec, sub_cs_prot in zip(spec.coord, cs.protonation.coords):
#                 print(sub_spec, sub_cs_prot, (sub_spec == sub_cs_prot))     
                
#         print("==========")     
#         for prot, cs in  zip(spec.protonation_states, spec.possible_cs) :
#              for sub_prot, sub_cs_prot in zip(prot.coords, cs.protonation.coords):
#                 print(sub_prot, sub_cs_prot, (sub_prot == sub_cs_prot))   
    else :
        spec.get_possible_cs()
        selected_cs.append(spec.possible_cs)   
    print("")
print(selected_cs)

LIGAND.SPLIT_LIGAND: self.indices=[0]
LIGAND.SPLIT_LIGAND: conn_idx=[0]
LIGAND.SPLIT_LIGAND: conn_labels=['Cl']
LIGAND.SPLIT_LIGAND: conn_coord=[[13.4461469, 6.6944761, 8.8941779]]
LIGAND.SPLIT_LIGAND: conn_radii=[1.02]
blocklist=[[0]]
LIGAND.SPLIT_LIGAND: block=[0]
ADD_ATOM: Metalist length 1
ADD_ATOM: Ligand Atoms 1
ADD_ATOM: site= 0
ADD_ATOM: evaluating apos=array([13.4461469,  6.6944761,  8.8941779]) and tgt.coord=[11.9462911, 8.1585379, 8.032954]
ADD_ATOM: received tmpconnec[posadded]=1
ADD_ATOM: Chosen Metal index None. H is added at site 0
protonation_states=[------------- Cell2mol Protonation ----------------
 Status                          = True
 Labels                          = ['Cl']
 Type                            = Local
 Atoms added in positions        = [0]
 Atoms blocked (no atoms added)  = [1]
---------------------------------------------------
]
smiles='[Cl-]'
smiles='[Cl-]'
smiles='[Cl-]'
smiles='[Cl-]'
smiles='[Cl-]'
Cl
spec.protonation_states=[------------- Cel

In [13]:
selected_cs

[[-1], [2, 3], [0], [0], [2, 3], [1]]

In [14]:
final_charge_distribution = balance_charge(newcell.unique_indices, newcell.unique_species, debug=2)

BALANCE: iterlist [[-1], [2, 3], [0], [0], [2, 3], [1]]
BALANCE: unique_indices [0, 0, 0, 0, 0, 0, 1, 1, 2, 3, 0, 0, 0, 4, 2, 3, 0, 0, 0, 4, 5, 5, 5, 5]
BALANCE: tmpdistr [(-1, 2, 0, 0, 2, 1), (-1, 2, 0, 0, 3, 1), (-1, 3, 0, 0, 2, 1), (-1, 3, 0, 0, 3, 1)]
BALANCE: alldistr added: [-1, -1, -1, -1, -1, -1, 2, 2, 0, 0, -1, -1, -1, 2, 0, 0, -1, -1, -1, 2, 1, 1, 1, 1]
d=[-1, -1, -1, -1, -1, -1, 2, 2, 0, 0, -1, -1, -1, 2, 0, 0, -1, -1, -1, 2, 1, 1, 1, 1]
BALANCE: alldistr added: [-1, -1, -1, -1, -1, -1, 2, 2, 0, 0, -1, -1, -1, 3, 0, 0, -1, -1, -1, 3, 1, 1, 1, 1]
d=[-1, -1, -1, -1, -1, -1, 2, 2, 0, 0, -1, -1, -1, 2, 0, 0, -1, -1, -1, 2, 1, 1, 1, 1]
d=[-1, -1, -1, -1, -1, -1, 2, 2, 0, 0, -1, -1, -1, 3, 0, 0, -1, -1, -1, 3, 1, 1, 1, 1]
BALANCE: alldistr added: [-1, -1, -1, -1, -1, -1, 3, 3, 0, 0, -1, -1, -1, 2, 0, 0, -1, -1, -1, 2, 1, 1, 1, 1]
d=[-1, -1, -1, -1, -1, -1, 2, 2, 0, 0, -1, -1, -1, 2, 0, 0, -1, -1, -1, 2, 1, 1, 1, 1]
d=[-1, -1, -1, -1, -1, -1, 2, 2, 0, 0, -1, -1, -1, 3, 0, 0, -1, -1

In [15]:
final_charge_distribution

[[-1,
  -1,
  -1,
  -1,
  -1,
  -1,
  2,
  2,
  0,
  0,
  -1,
  -1,
  -1,
  2,
  0,
  0,
  -1,
  -1,
  -1,
  2,
  1,
  1,
  1,
  1]]

In [16]:
def set_charges_create_bonds (specie, unique_indices, unique_species, final_charge_distribution):
        
    spec = unique_species[specie.unique_index]
    indices = [index for index, value in enumerate(unique_indices) if value == specie.unique_index]
    target_charge = [final_charge_distribution[i] for i in indices][0] 
    if debug > 1: print(spec, indices, target_charge)
    
    if (specie.subtype == "molecule" and specie.iscomplex == False) or (specie.subtype == "ligand"):
        formula = specie.formula
        charge_list = [cs.corr_total_charge for cs in spec.possible_cs]
    
    elif specie.subtype == "metal":
        formula = specie.label
        charge_list = spec.possible_cs
    
    if target_charge in charge_list:           
        if debug > 1: print(f"Target charge {target_charge} of {formula} exists in {charge_list}." )
    else:
        if debug > 1: print(f"ERROR: Target charge {target_charge} of {formula} does not exist in {charge_list}." )
        return None
        
    if (specie.subtype == "molecule" and specie.iscomplex == False) or (specie.subtype == "ligand"):
        print(specie.formula)
        specie.get_protonation_states(debug=0)
        specie.get_possible_cs(debug=0)
        formula = specie.formula
        charge_list = [cs.corr_total_charge for cs in specie.possible_cs]        
        
        if target_charge in charge_list:
            if debug > 1: print(f"Target charge {target_charge} of {formula} exists in {charge_list}.")
            idx = charge_list.index(target_charge)
            cs = specie.possible_cs[idx]
            prot = cs.protonation
            specie.set_charges(cs.corr_total_charge, cs.corr_atom_charges, cs.smiles, cs.rdkit_obj)
            specie.create_bonds(debug=0)
        else:
            if debug > 1: print(f"ERROR: Target charge {target_charge} of {formula} does not exist in {charge_list}." )
            return None
                    
    elif specie.subtype == "metal":
        print(specie.label)
        specie.get_possible_cs(debug=0)
        formula = specie.label
        charge_list = spec.possible_cs        

        if target_charge in charge_list:
            if debug > 1: print(f"Target charge {target_charge} of {formula} exists in {charge_list}." )
            idx = charge_list.index(target_charge)
            cs = specie.possible_cs[idx]
            specie.set_charge(cs)         
        else:
            if debug > 1: print(f"ERROR: Target charge {target_charge} of {formula} does not exist in {charge_list}." )
            return None  

In [17]:
def prepare_mols_v4 (moleclist: list, unique_indices: list, unique_species: list, 
                      selected_cs: list, final_charge_distribution: list, debug: int=0):
    count = 0 
    for mol in moleclist:
        if mol.iscomplex == False:
            set_charges_create_bonds(mol, unique_indices, unique_species, final_charge_distribution)
            count += 1
        
        elif mol.iscomplex:
            tmp_atcharge = np.zeros((mol.natoms))
            tmp_smiles = []
            
            for lig in mol.ligands:            
                set_charges_create_bonds(lig, unique_indices, unique_species, final_charge_distribution)
                count += 1
                 
                tmp_smiles.append(lig.smiles)
                parent_indices = lig.get_parent_indices("molecule")
                for kdx, a in enumerate(parent_indices):
                    tmp_atcharge[a] = lig.atomic_charges[kdx]
                    
            for met in mol.metals:        
                set_charges_create_bonds(met, unique_indices, unique_species, final_charge_distribution)
                count += 1
                parent_index = met.get_parent_index("molecule")
                tmp_atcharge[parent_index] = met.charge     
                
            mol.set_charges(int(sum(tmp_atcharge)), atomic_charges=tmp_atcharge, smiles=tmp_smiles)

    if count != len(final_charge_distribution):
        Warning = True
    else:
        Warning = False
    
    return moleclist, Warning
                
                
#                     ref_data, target_data = arrange_data_for_reorder(spec, mol)
#                     print(f"{spec.labels=}")
#                     print(f"{mol.labels=}")
#                     print(ref_data, target_data)
#                     dummy1, dummy2, map12 = reorder(ref_data, target_data, spec.coord, mol.coord)
                    
#                     print("*****Before function reorder within class****")
#                     print(cs.protonation)
#                     prot = cs.protonation.reorder(map12, debug=debug)
#                     print("*****After function reorder within class****")
#                     print(cs.protonation)
#                     print(prot)
#                     ref_data, target_data = arrange_data_for_reorder(mol, spec)
#                     print(f"{spec.labels=}")
#                     print(f"{mol.labels=}")
#                     print(ref_data, target_data)
#                     dummy1, dummy2, map12 = reorder(ref_data, target_data, mol.coord, spec.coord)
#                     reordered_prot_spec = reorder_protonation(spec.possible_cs[0].protonation, map12, debug=debug)                    
#                     print("**************** After reorder **************** mol vs reordered_protonation")    
#                     for l_mol, c_mol, l_prot, c_prot in zip(mol.labels, mol.coord, prot.labels, prot.coords):
#                         print(l_mol, l_prot, (l_mol == l_prot), c_mol, c_prot, (c_mol==c_prot) )   
#                     print("**************** After reorder **************** mol vs reordered_protonation_spec") 
#                     for l_mol, c_mol, l_prot, c_prot in zip(mol.labels, mol.coord, reordered_prot_spec.labels, reordered_prot_spec.coords):
#                         print(l_mol, l_prot, (l_mol == l_prot), c_mol, c_prot, (c_mol==c_prot) )  

In [18]:
newcell.moleclist, newcell.error_prepare_mols = prepare_mols_v4(newcell.moleclist, 
                                                                newcell.unique_indices, 
                                                                newcell.unique_species, 
                                                                selected_cs, 
                                                                final_charge_distribution[0], 
                                                                debug=debug)

Cl
protonation_states=[------------- Cell2mol Protonation ----------------
 Status                          = True
 Labels                          = ['Cl']
 Type                            = Local
 Atoms added in positions        = [0]
 Atoms blocked (no atoms added)  = [1]
---------------------------------------------------
]
smiles='[Cl-]'
smiles='[Cl-]'
smiles='[Cl-]'
smiles='[Cl-]'
smiles='[Cl-]'
Cl
ADD_ATOM: Metalist length 1
ADD_ATOM: Ligand Atoms 1
ADD_ATOM: site= 0
ADD_ATOM: evaluating apos=array([7.4116689, 8.4886261, 6.9897009]) and tgt.coord=[8.9115248, 7.0245642, 7.8509248]
ADD_ATOM: received tmpconnec[posadded]=1
ADD_ATOM: Chosen Metal index None. H is added at site 0
protonation_states=[------------- Cell2mol Protonation ----------------
 Status                          = True
 Labels                          = ['Cl']
 Type                            = Local
 Atoms added in positions        = [0]
 Atoms blocked (no atoms added)  = [1]
------------------------------------

smiles='[H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[N+](C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])(C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]'
smiles='[H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[N+](C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])(C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]'
smiles='[H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[N+](C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])(C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]'
smiles='[H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[N+](C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])(C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]'
smiles='[H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[N+](C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])(C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C([H])([

In [19]:
newcell.error_prepare_mols

False

In [20]:
newcell.moleclist

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 8
  Formula                      = Cl6-Fe2
  Has Adjacency Matrix         = YES
  Total Charge                 = -2
  Smiles                       = ['[Cl-]', '[Cl-]', '[Cl-]', '[Cl-]', '[Cl-]', '[Cl-]']
  Origin                       = cell.reconstruct
  Number of Ligands            = 6
  Number of Metals             = 2
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 15
  Formula                      = H10-C4-O
  Has Adjacency Matrix         = YES
  Total Charge                 = 0
  Smiles                       = [H]C([H])([H])C([H])([H])OC([H])([H])C([H])([H])[H]
  Orig

In [21]:
newcell.assign_spin(debug=2)

------------- Cell2mol GROUP Object --------------
 Version                      = 0.1
 Type                         = specie
 Sub-Type                     = group
 Number of Atoms              = 1
 Formula                      = Cl
 Has Adjacency Matrix         = YES
 Origin                       = split_ligand
 Number of Metals             = 1
---------------------------------------------------

['Fe', 'Cl'] [[8.9115248, 7.0245642, 7.8509248], [13.4461469, 6.6944761, 8.8941779]]
------------- Cell2mol GROUP Object --------------
 Version                      = 0.1
 Type                         = specie
 Sub-Type                     = group
 Number of Atoms              = 1
 Formula                      = Cl
 Has Adjacency Matrix         = YES
 Origin                       = split_ligand
 Number of Metals             = 1
---------------------------------------------------

['Fe', 'Cl'] [[8.9115248, 7.0245642, 7.8509248], [7.4116689, 8.4886261, 6.9897009]]
------------- Cell2mol GROUP 

[------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 8
  Formula                      = Cl6-Fe2
  Has Adjacency Matrix         = YES
  Total Charge                 = -2
  Spin                         = None
  Smiles                       = ['[Cl-]', '[Cl-]', '[Cl-]', '[Cl-]', '[Cl-]', '[Cl-]']
  Origin                       = cell.reconstruct
  Number of Ligands            = 6
  Number of Metals             = 2
 ---------------------------------------------------,
 ------------- Cell2mol MOLECULE Object --------------
  Version                      = 0.1
  Type                         = specie
  Sub-Type                     = molecule
  Number of Atoms              = 15
  Formula                      = H10-C4-O
  Has Adjacency Matrix         = YES
  Total Charge                 = 0
  Spin                         = 1
  Smiles          

In [22]:
for mol in newcell.moleclist:
    if mol.iscomplex:
        for met in mol.metals:
            print(met.label, met.spin)

Fe 5
Fe 5
Fe 5
Fe 5


In [23]:
newcell.save("INOVAL/INOVAL.cell")

SAVING cell2mol CELL object to INOVAL/INOVAL.cell
